# Trendyol CatBoost GPU With Train-Term Negatives

This Kaggle notebook is self-contained. Run each cell normally. It writes the required Python modules into `/kaggle/working/src`, builds query-category priors, mines negatives from `training_pairs.csv` terms, trains CatBoost on GPU, predicts `submission_pairs.csv`, and writes rate-based submissions.

Inputs required from Kaggle:
- `items.csv`
- `terms.csv`
- `training_pairs.csv`
- `submission_pairs.csv`

The important correction is that negatives are mined from training terms and the item catalog, not from test submission pairs.


In [ ]:
from pathlib import Path
import argparse
import importlib
import os
import sys

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
SRC_DIR = WORK_DIR / "src"
OUT_DIR = WORK_DIR / "outputs"
CAT_OUT_DIR = OUT_DIR / "category_aware_notebook"
SRC_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
CAT_OUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

REQUIRED_FILES = {
    "items.csv",
    "terms.csv",
    "training_pairs.csv",
    "submission_pairs.csv",
}


def find_data_dir() -> Path:
    candidates = []
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend([p for p in input_root.rglob("*") if p.is_dir()])
    candidates.append(Path.cwd() / "data")
    candidates.append(Path.cwd())

    for path in candidates:
        if all((path / name).exists() for name in REQUIRED_FILES):
            return path
    raise FileNotFoundError(
        "Could not find a directory containing all required competition CSV files."
    )


DATA_DIR = find_data_dir()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("DATA_DIR =", DATA_DIR)
print("WORK_DIR =", WORK_DIR)
print("SRC_DIR =", SRC_DIR)

In [ ]:
# Dependency check. If this fails on Kaggle, enable Internet and install the missing package once in a separate setup cell.
missing = []
for package, import_name in [
    ("catboost", "catboost"),
    ("lightgbm", "lightgbm"),
    ("rapidfuzz", "rapidfuzz"),
    ("text-unidecode", "text_unidecode"),
    ("scikit-learn", "sklearn"),
    ("scipy", "scipy"),
]:
    try:
        __import__(import_name)
    except ImportError:
        missing.append(package)

if missing:
    raise ImportError("Missing packages: " + ", ".join(missing))
print("dependencies ok")

## Create `lexical_baseline.py`


In [ ]:
%%writefile src/lexical_baseline.py
from __future__ import annotations

import argparse
import csv
import math
import re
from collections import Counter
from functools import lru_cache
from pathlib import Path

import pandas as pd
from rapidfuzz import fuzz
from text_unidecode import unidecode
from tqdm import tqdm


TOKEN_RE = re.compile(r"[a-z0-9]+")

STOPWORDS = {
    "a",
    "ve",
    "ile",
    "icin",
    "bir",
    "adet",
    "ad",
    "li",
    "set",
    "seti",
    "takim",
    "takimi",
    "urun",
    "model",
    "uyumlu",
    "orjinal",
    "orijinal",
}

COLORS = {
    "beyaz",
    "siyah",
    "kirmizi",
    "mavi",
    "yesil",
    "sari",
    "pembe",
    "mor",
    "gri",
    "lacivert",
    "kahverengi",
    "bej",
    "krem",
    "turuncu",
    "bordo",
    "gold",
    "altin",
    "gumus",
    "haki",
    "taba",
    "lila",
    "ekru",
    "vizon",
    "füme",
}

GENDER_ALLOWED = {
    "kadin": {"kadin", "unisex", "unknown"},
    "bayan": {"kadin", "unisex", "unknown"},
    "erkek": {"erkek", "unisex", "unknown"},
    "unisex": {"kadin", "erkek", "unisex", "unknown"},
}

AGE_ALLOWED = {
    "bebek": {"bebek", "unknown"},
    "cocuk": {"cocuk", "bebek", "unknown"},
    "kiz": {"cocuk", "bebek", "unknown"},
}


def normalize_text(value: object) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    text = unidecode(str(value).lower())
    text = text.replace("&", " ")
    return re.sub(r"\s+", " ", text).strip()


def stem_token(token: str) -> str:
    if len(token) <= 4:
        return token

    for suffix in (
        "larin",
        "lerin",
        "lari",
        "leri",
        "lara",
        "lere",
        "larda",
        "lerde",
        "ndan",
        "nden",
        "dan",
        "den",
        "nin",
        "nun",
        "ler",
        "lar",
        "lik",
        "luk",
        "cuk",
        "cik",
        "ci",
        "cu",
        "si",
        "su",
        "u",
        "i",
    ):
        if token.endswith(suffix) and len(token) - len(suffix) >= 3:
            return token[: -len(suffix)]
    return token


@lru_cache(maxsize=1_000_000)
def token_set(text: str) -> frozenset[str]:
    out: set[str] = set()
    for token in TOKEN_RE.findall(text):
        if token in STOPWORDS:
            continue
        out.add(token)
        stemmed = stem_token(token)
        if stemmed not in STOPWORDS:
            out.add(stemmed)
    return frozenset(out)


def query_tokens(text: str) -> tuple[str, ...]:
    seen: set[str] = set()
    out: list[str] = []
    for token in TOKEN_RE.findall(text):
        if token in STOPWORDS:
            continue
        stemmed = stem_token(token)
        for candidate in (token, stemmed):
            if candidate and candidate not in STOPWORDS and candidate not in seen:
                seen.add(candidate)
                out.append(candidate)
                break
    return tuple(out)


def coverage(query: tuple[str, ...], field: frozenset[str]) -> float:
    if not query:
        return 0.0
    return sum(1 for token in query if token in field) / len(query)


def color_tokens(query: tuple[str, ...]) -> set[str]:
    return {token for token in query if token in COLORS}


def load_terms(path: Path) -> dict[str, dict[str, object]]:
    terms = pd.read_csv(path)
    records: dict[str, dict[str, object]] = {}
    for row in terms.itertuples(index=False):
        norm_query = normalize_text(row.query)
        q_tokens = query_tokens(norm_query)
        records[row.term_id] = {
            "query": norm_query,
            "tokens": q_tokens,
            "colors": color_tokens(q_tokens),
        }
    return records


def load_items(path: Path) -> dict[str, tuple[str, str, str, str, str, str]]:
    items: dict[str, tuple[str, str, str, str, str, str]] = {}
    with path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        for row in tqdm(reader, desc="items"):
            item_id = row["item_id"]
            items[item_id] = (
                normalize_text(row.get("title")),
                normalize_text(row.get("category")),
                normalize_text(row.get("brand")),
                normalize_text(row.get("gender")) or "unknown",
                normalize_text(row.get("age_group")) or "unknown",
                normalize_text(row.get("attributes")),
            )
    return items


def pair_features(term: dict[str, object], item: tuple[str, str, str, str, str, str]) -> dict[str, float]:
    query = term["query"]  # type: ignore[index]
    q_tokens = term["tokens"]  # type: ignore[index]
    q_colors = term["colors"]  # type: ignore[index]
    title, category, brand, gender, age_group, attrs = item

    title_tokens = token_set(title)
    category_tokens = token_set(category)
    brand_tokens = token_set(brand)
    attr_tokens = token_set(attrs)
    full_tokens = title_tokens | category_tokens | brand_tokens | attr_tokens

    title_cov = coverage(q_tokens, title_tokens)
    category_cov = coverage(q_tokens, category_tokens)
    brand_cov = coverage(q_tokens, brand_tokens)
    attr_cov = coverage(q_tokens, attr_tokens)
    full_cov = coverage(q_tokens, full_tokens)

    title_ratio = fuzz.token_set_ratio(query, title) / 100.0
    category_ratio = fuzz.token_set_ratio(query, category) / 100.0
    brand_ratio = max(
        fuzz.ratio(query, brand) / 100.0,
        fuzz.partial_ratio(query, brand) / 100.0 if brand else 0.0,
    )

    query_in_title = float(bool(query and query in title))
    query_in_category = float(bool(query and query in category))
    query_in_brand = float(bool(query and query in brand))
    full_complete = float(full_cov >= 0.999)
    full_high = float(0.75 <= full_cov < 0.999)
    title_complete = float(title_cov >= 0.999)
    category_complete = float(category_cov >= 0.999)
    brand_complete = float(brand_cov >= 0.999 and bool(brand))
    short_complete = float(len(q_tokens) <= 2 and full_cov >= 0.999)

    gender_mismatch = 0.0
    for gender_word, allowed in GENDER_ALLOWED.items():
        if gender_word in q_tokens and gender not in allowed and gender_word not in full_tokens:
            gender_mismatch = 1.0
            break

    age_mismatch = 0.0
    for age_word, allowed in AGE_ALLOWED.items():
        if age_word in q_tokens and age_group not in allowed and age_word not in full_tokens:
            age_mismatch = 1.0
            break

    color_mismatch = float(bool(q_colors and full_tokens.isdisjoint(q_colors)))
    weak_match = float(full_cov < 0.34 and max(title_ratio, category_ratio, brand_ratio) < 0.55)

    features = {
        "n_query_tokens": float(len(q_tokens)),
        "title_cov": title_cov,
        "category_cov": category_cov,
        "brand_cov": brand_cov,
        "attr_cov": attr_cov,
        "full_cov": full_cov,
        "title_ratio": title_ratio,
        "category_ratio": category_ratio,
        "brand_ratio": brand_ratio,
        "query_in_title": query_in_title,
        "query_in_category": query_in_category,
        "query_in_brand": query_in_brand,
        "full_complete": full_complete,
        "full_high": full_high,
        "title_complete": title_complete,
        "category_complete": category_complete,
        "brand_complete": brand_complete,
        "short_complete": short_complete,
        "gender_mismatch": gender_mismatch,
        "age_mismatch": age_mismatch,
        "color_mismatch": color_mismatch,
        "weak_match": weak_match,
        "title_len": float(len(title_tokens)),
        "category_len": float(len(category_tokens)),
        "brand_len": float(len(brand_tokens)),
        "attr_len": float(len(attr_tokens)),
    }

    features["lexical_score"] = lexical_score_from_features(features)
    return features


def lexical_score_from_features(features: dict[str, float]) -> float:
    title_cov = features["title_cov"]
    category_cov = features["category_cov"]
    brand_cov = features["brand_cov"]
    attr_cov = features["attr_cov"]
    full_cov = features["full_cov"]
    title_ratio = features["title_ratio"]
    category_ratio = features["category_ratio"]
    brand_ratio = features["brand_ratio"]

    score = 0.0
    score += 2.1 * full_cov
    score += 1.4 * title_cov
    score += 1.0 * category_cov
    score += 1.2 * brand_cov
    score += 0.55 * attr_cov
    score += 0.75 * title_ratio
    score += 0.45 * category_ratio
    score += 0.55 * brand_ratio

    if features["query_in_title"]:
        score += 1.0
    if features["query_in_category"]:
        score += 0.8
    if features["query_in_brand"]:
        score += 1.1
    if features["full_complete"]:
        score += 1.2
    elif features["full_high"]:
        score += 0.55
    if features["title_complete"]:
        score += 0.9
    if features["category_complete"]:
        score += 0.7
    if features["brand_complete"]:
        score += 0.9
    if features["short_complete"]:
        score += 0.55

    if features["gender_mismatch"]:
        score -= 0.85
    if features["age_mismatch"]:
        score -= 0.75
    if features["color_mismatch"]:
        score -= 0.7
    if features["weak_match"]:
        score -= 0.9

    return score


def score_pair(term: dict[str, object], item: tuple[str, str, str, str, str, str]) -> float:
    return pair_features(term, item)["lexical_score"]


def score_pairs(args: argparse.Namespace) -> None:
    data_dir = Path(args.data_dir)
    out_path = Path(args.output)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    terms = load_terms(data_dir / "terms.csv")
    items = load_items(data_dir / "items.csv")

    in_path = data_dir / args.pairs
    total = args.limit
    if total is None:
        with in_path.open(newline="", encoding="utf-8") as handle:
            total = sum(1 for _ in handle) - 1

    with in_path.open(newline="", encoding="utf-8") as in_handle, out_path.open(
        "w", newline="", encoding="utf-8"
    ) as out_handle:
        reader = csv.DictReader(in_handle)
        writer = csv.writer(out_handle)
        writer.writerow(["id", "term_id", "score"])
        for i, row in enumerate(tqdm(reader, total=total, desc="pairs")):
            if args.limit is not None and i >= args.limit:
                break
            item = items[row["item_id"]]
            term = terms[row["term_id"]]
            writer.writerow([row["id"], row["term_id"], f"{score_pair(term, item):.6f}"])


def summarize_scores(args: argparse.Namespace) -> None:
    scores = pd.read_csv(args.scores)
    print(scores["score"].describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
    for threshold in args.thresholds:
        pred = scores["score"] >= threshold
        print(f"threshold={threshold:.3f} positives={int(pred.sum())} rate={pred.mean():.5f}")

    by_term = scores.groupby("term_id")["score"].agg(["count", "max", "mean"])
    print(by_term.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))


def make_submission(args: argparse.Namespace) -> None:
    scores = pd.read_csv(args.scores)
    pred = scores["score"] >= args.threshold

    if args.min_per_term > 0 or args.max_per_term > 0:
        scores = scores.assign(pred=pred.astype("int8"))
        parts = []
        for _, group in tqdm(scores.groupby("term_id", sort=False), desc="terms"):
            group = group.copy()
            order = group["score"].sort_values(ascending=False).index
            if args.min_per_term > 0 and group.loc[order[: args.min_per_term], "score"].max() >= args.min_score:
                group.loc[order[: args.min_per_term], "pred"] = 1
            if args.max_per_term > 0:
                positives = group.index[group["pred"] == 1]
                if len(positives) > args.max_per_term:
                    keep = group.loc[positives].sort_values("score", ascending=False).head(args.max_per_term).index
                    group.loc[positives, "pred"] = 0
                    group.loc[keep, "pred"] = 1
            parts.append(group[["id", "pred"]])
        scored = pd.concat(parts, ignore_index=True)
        pred_values = scored["pred"].astype("int8")
        ids = scored["id"]
    else:
        pred_values = pred.astype("int8")
        ids = scores["id"]

    output = pd.DataFrame({"id": ids, "prediction": pred_values})
    output.to_csv(args.output, index=False)
    counts = Counter(output["prediction"])
    print(f"wrote {args.output}")
    print(f"rows={len(output)} positives={counts[1]} rate={counts[1] / len(output):.5f}")


def main() -> None:
    parser = argparse.ArgumentParser()
    sub = parser.add_subparsers(dest="cmd", required=True)

    score = sub.add_parser("score")
    score.add_argument("--data-dir", default="data")
    score.add_argument("--pairs", default="submission_pairs.csv")
    score.add_argument("--output", default="outputs/lexical_scores.csv")
    score.add_argument("--limit", type=int)
    score.set_defaults(func=score_pairs)

    summary = sub.add_parser("summary")
    summary.add_argument("--scores", default="outputs/lexical_scores.csv")
    summary.add_argument("--thresholds", type=float, nargs="+", default=[3.0, 3.5, 4.0, 4.5, 5.0])
    summary.set_defaults(func=summarize_scores)

    submit = sub.add_parser("submit")
    submit.add_argument("--scores", default="outputs/lexical_scores.csv")
    submit.add_argument("--output", default="outputs/submission_lexical.csv")
    submit.add_argument("--threshold", type=float, required=True)
    submit.add_argument("--min-per-term", type=int, default=0)
    submit.add_argument("--max-per-term", type=int, default=0)
    submit.add_argument("--min-score", type=float, default=0.0)
    submit.set_defaults(func=make_submission)

    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


## Create `train_term_negatives.py`


In [ ]:
%%writefile src/train_term_negatives.py
from __future__ import annotations

import argparse
import csv
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm

from lexical_baseline import normalize_text


def item_search_text(items: pd.DataFrame) -> pd.Series:
    title = items["title"].map(normalize_text)
    category = items["category"].map(normalize_text)
    brand = items["brand"].map(normalize_text)
    attrs = items["attributes"].map(normalize_text)
    return (
        title
        + " "
        + title
        + " "
        + brand
        + " "
        + brand
        + " "
        + category
        + " "
        + attrs
    ).str.strip()


def pair_key(frame: pd.DataFrame) -> pd.Series:
    return frame["term_id"].astype(str) + "\t" + frame["item_id"].astype(str)


def topk_sparse_row(row: sparse.spmatrix, k: int) -> tuple[np.ndarray, np.ndarray]:
    row = row.tocsr()
    indices = row.indices
    data = row.data
    if len(data) == 0:
        return indices, data

    if len(data) > k:
        selected = np.argpartition(data, -k)[-k:]
        selected = selected[np.argsort(-data[selected], kind="mergesort")]
    else:
        selected = np.argsort(-data, kind="mergesort")
    return indices[selected], data[selected]


def sample_easy_items(
    item_count: int,
    excluded: set[int],
    n_items: int,
    rng: np.random.Generator,
) -> list[int]:
    if n_items <= 0:
        return []

    selected: list[int] = []
    attempts = 0
    max_attempts = max(100, n_items * 50)
    while len(selected) < n_items and attempts < max_attempts:
        item_pos = int(rng.integers(0, item_count))
        attempts += 1
        if item_pos in excluded:
            continue
        excluded.add(item_pos)
        selected.append(item_pos)
    return selected


def load_positive_pairs(path: Path) -> pd.DataFrame:
    pairs = pd.read_csv(path, dtype=str, keep_default_na=False)
    if "label" in pairs.columns:
        pairs = pairs.loc[pairs["label"].astype(str).ne("0")]
    pairs = pairs[["term_id", "item_id"]].drop_duplicates().reset_index(drop=True)
    if pairs.empty:
        raise ValueError(f"No positive training pairs found in {path}")
    return pairs


def mine_negatives(args: argparse.Namespace) -> None:
    data_dir = Path(args.data_dir)
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    terms = pd.read_csv(data_dir / "terms.csv", dtype=str, keep_default_na=False)
    items = pd.read_csv(
        data_dir / "items.csv",
        dtype=str,
        keep_default_na=False,
        usecols=["item_id", "title", "category", "brand", "attributes"],
    )
    positives = load_positive_pairs(data_dir / "training_pairs.csv")

    train_term_ids = positives["term_id"].drop_duplicates().to_numpy()
    if args.limit_terms:
        train_term_ids = train_term_ids[: args.limit_terms]
        positives = positives.loc[positives["term_id"].isin(train_term_ids)].reset_index(drop=True)
    if args.limit_items:
        items = items.head(args.limit_items).reset_index(drop=True)

    terms = terms.loc[terms["term_id"].isin(train_term_ids), ["term_id", "query"]].copy()
    missing_terms = set(train_term_ids) - set(terms["term_id"])
    if missing_terms:
        examples = sorted(missing_terms)[:5]
        raise KeyError(f"{len(missing_terms)} training term_ids are missing from terms.csv, examples={examples}")

    terms["query_norm"] = terms["query"].map(normalize_text)
    terms = terms.set_index("term_id").loc[train_term_ids].reset_index()
    item_ids = items["item_id"].astype(str).to_numpy()
    item_to_pos = {item_id: i for i, item_id in enumerate(item_ids)}

    positive_by_term: dict[str, set[int]] = {}
    for term_id, group in positives.groupby("term_id", sort=False):
        positions = {item_to_pos[item_id] for item_id in group["item_id"].astype(str) if item_id in item_to_pos}
        positive_by_term[str(term_id)] = positions

    item_text = item_search_text(items)
    corpus = pd.concat([terms["query_norm"], item_text], ignore_index=True)
    vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, args.word_ngram_max),
        min_df=args.min_df,
        max_features=args.max_features,
        dtype=np.float32,
        norm="l2",
        sublinear_tf=True,
    )
    vectorizer.fit(corpus)
    term_matrix = vectorizer.transform(terms["query_norm"])
    item_matrix = vectorizer.transform(item_text)
    item_matrix_t = item_matrix.T.tocsr()
    print(
        f"terms={len(terms):,} items={len(items):,} positives={len(positives):,} "
        f"vocab={len(vectorizer.vocabulary_):,}"
    )

    rng = np.random.default_rng(args.seed)
    counts = {"hard": 0, "mid": 0, "easy": 0}
    total = 0
    positive_keys = set(pair_key(positives))

    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["term_id", "item_id", "negative_band", "tfidf_score"])

        for start in tqdm(range(0, len(terms), args.chunk_size), desc="mine train-term negatives"):
            stop = min(start + args.chunk_size, len(terms))
            scores = term_matrix[start:stop] @ item_matrix_t
            scores = scores.tocsr()

            for row_offset, term_id in enumerate(terms["term_id"].iloc[start:stop].astype(str)):
                pos_items = set(positive_by_term.get(term_id, set()))
                excluded = set(pos_items)
                row_indices, row_scores = topk_sparse_row(scores.getrow(row_offset), args.retrieve_topk)

                candidates: list[tuple[int, float]] = []
                for item_pos, score in zip(row_indices, row_scores):
                    item_pos = int(item_pos)
                    if item_pos in excluded:
                        continue
                    excluded.add(item_pos)
                    candidates.append((item_pos, float(score)))

                hard = candidates[: args.hard_per_term]
                mid_pool = candidates[args.hard_per_term :]
                if len(mid_pool) > args.mid_per_term:
                    chosen = rng.choice(len(mid_pool), size=args.mid_per_term, replace=False)
                    mid = [mid_pool[int(i)] for i in chosen]
                else:
                    mid = mid_pool

                rows: list[tuple[str, str, str, float]] = []
                for item_pos, score in hard:
                    rows.append((term_id, item_ids[item_pos], "hard", score))
                for item_pos, score in mid:
                    rows.append((term_id, item_ids[item_pos], "mid", score))

                easy_positions = sample_easy_items(len(item_ids), excluded, args.easy_per_term, rng)
                for item_pos in easy_positions:
                    rows.append((term_id, item_ids[item_pos], "easy", 0.0))

                for out_term_id, item_id, band, score in rows:
                    if f"{out_term_id}\t{item_id}" in positive_keys:
                        continue
                    writer.writerow([out_term_id, item_id, band, f"{score:.8f}"])
                    counts[band] += 1
                    total += 1

    print(f"wrote {output_path}")
    print(f"negatives={total:,} hard={counts['hard']:,} mid={counts['mid']:,} easy={counts['easy']:,}")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", default="data")
    parser.add_argument("--output", default="outputs/train_term_negatives.csv")
    parser.add_argument("--hard-per-term", type=int, default=25)
    parser.add_argument("--mid-per-term", type=int, default=15)
    parser.add_argument("--easy-per-term", type=int, default=10)
    parser.add_argument("--retrieve-topk", type=int, default=300)
    parser.add_argument("--chunk-size", type=int, default=128)
    parser.add_argument("--max-features", type=int, default=250_000)
    parser.add_argument("--min-df", type=int, default=2)
    parser.add_argument("--word-ngram-max", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--limit-terms", type=int, help="Debug only: mine negatives for the first N train terms.")
    parser.add_argument("--limit-items", type=int, help="Debug only: restrict the item catalog.")
    args = parser.parse_args()
    mine_negatives(args)


if __name__ == "__main__":
    main()


## Create `pu_catboost.py`


In [ ]:
%%writefile src/pu_catboost.py
from __future__ import annotations

import argparse
import csv
import gc
import math
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from lexical_baseline import load_items, load_terms, normalize_text, pair_features


BASE_FEATURE_NAMES = [
    "n_query_tokens",
    "title_cov",
    "category_cov",
    "brand_cov",
    "attr_cov",
    "full_cov",
    "title_ratio",
    "category_ratio",
    "brand_ratio",
    "query_in_title",
    "query_in_category",
    "query_in_brand",
    "full_complete",
    "full_high",
    "title_complete",
    "category_complete",
    "brand_complete",
    "short_complete",
    "gender_mismatch",
    "age_mismatch",
    "color_mismatch",
    "weak_match",
    "title_len",
    "category_len",
    "brand_len",
    "attr_len",
    "lexical_score",
]

CATEGORY_LEVELS = ("cat_l1", "cat_l2")
CATEGORY_TOPK = 5
CATEGORY_FEATURE_NAMES: list[str] = []
for level in CATEGORY_LEVELS:
    CATEGORY_FEATURE_NAMES.extend(
        [
            f"{level}_top1_match",
            f"{level}_top3_match",
            f"{level}_top5_match",
            f"{level}_candidate_prob",
            f"{level}_top1_prob",
            f"{level}_rank",
        ]
    )

ITEM_CAT_FEATURES = ["item_cat_l1", "item_cat_l2", "item_gender", "item_age_group"]
FEATURE_NAMES = BASE_FEATURE_NAMES + CATEGORY_FEATURE_NAMES + ITEM_CAT_FEATURES
CAT_FEATURES = ITEM_CAT_FEATURES


def split_category(value: object) -> tuple[str, str, str, str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        parts = ["unknown"]
    else:
        parts = [normalize_text(part) for part in str(value).split("/")]
        parts = [part for part in parts if part]
        if not parts:
            parts = ["unknown"]

    cat_l1 = parts[0]
    cat_l2 = "/".join(parts[:2]) if len(parts) >= 2 else cat_l1
    cat_l3 = "/".join(parts[:3]) if len(parts) >= 3 else cat_l2
    cat_leaf = "/".join(parts)
    return cat_l1, cat_l2, cat_l3, cat_leaf


def add_category_levels(frame: pd.DataFrame, category_col: str = "category") -> pd.DataFrame:
    levels = pd.DataFrame(
        frame[category_col].map(split_category).tolist(),
        columns=["cat_l1", "cat_l2", "cat_l3", "cat_leaf"],
        index=frame.index,
    )
    return pd.concat([frame, levels], axis=1)


def build_term_category_lookup(
    topk: pd.DataFrame,
    levels: tuple[str, ...] = CATEGORY_LEVELS,
    k: int = CATEGORY_TOPK,
) -> dict[str, dict[str, object]]:
    lookup: dict[str, dict[str, object]] = {}
    for row in topk.itertuples(index=False):
        row_dict = row._asdict()
        entry: dict[str, object] = {}
        for level in levels:
            scores: dict[str, float] = {}
            for rank in range(1, k + 1):
                label = row_dict.get(f"{level}_top{rank}")
                score = row_dict.get(f"{level}_prob{rank}")
                if isinstance(label, str) and label:
                    scores[label] = float(score)
            entry[level] = scores
            entry[f"{level}_top1_prob"] = next(iter(scores.values()), 0.0)
        lookup[row_dict["term_id"]] = entry
    return lookup


def build_item_meta_lookup(items_path: Path) -> dict[str, dict[str, str]]:
    items = pd.read_csv(items_path, usecols=["item_id", "category", "gender", "age_group"])
    items = add_category_levels(items)
    lookup: dict[str, dict[str, str]] = {}
    for row in tqdm(items.itertuples(index=False), total=len(items), desc="item metadata"):
        lookup[row.item_id] = {
            "cat_l1": row.cat_l1,
            "cat_l2": row.cat_l2,
            "item_cat_l1": row.cat_l1 or "unknown",
            "item_cat_l2": row.cat_l2 or "unknown",
            "item_gender": normalize_text(row.gender) or "unknown",
            "item_age_group": normalize_text(row.age_group) or "unknown",
        }
    return lookup


def category_pair_features(
    term_id: str,
    item_id: str,
    term_category_lookup: dict[str, dict[str, object]],
    item_meta_lookup: dict[str, dict[str, str]],
) -> dict[str, float | str]:
    term_entry = term_category_lookup.get(term_id, {})
    item_entry = item_meta_lookup.get(item_id, {})
    out: dict[str, float | str] = {
        "item_cat_l1": item_entry.get("item_cat_l1", "unknown"),
        "item_cat_l2": item_entry.get("item_cat_l2", "unknown"),
        "item_gender": item_entry.get("item_gender", "unknown"),
        "item_age_group": item_entry.get("item_age_group", "unknown"),
    }

    for level in CATEGORY_LEVELS:
        scores = term_entry.get(level, {})
        if not isinstance(scores, dict):
            scores = {}
        ordered_labels = list(scores.keys())
        candidate = item_entry.get(level, "")
        rank = ordered_labels.index(candidate) + 1 if candidate in scores else 0

        out[f"{level}_top1_match"] = float(rank == 1)
        out[f"{level}_top3_match"] = float(1 <= rank <= 3)
        out[f"{level}_top5_match"] = float(1 <= rank <= 5)
        out[f"{level}_candidate_prob"] = float(scores.get(candidate, 0.0))
        out[f"{level}_top1_prob"] = float(term_entry.get(f"{level}_top1_prob", 0.0))
        out[f"{level}_rank"] = float(rank)

    return out


def rows_to_frame(
    rows: pd.DataFrame,
    terms: dict[str, dict[str, object]],
    items: dict[str, tuple[str, str, str, str, str, str]],
    term_category_lookup: dict[str, dict[str, object]],
    item_meta_lookup: dict[str, dict[str, str]],
    desc: str,
) -> pd.DataFrame:
    numeric_names = BASE_FEATURE_NAMES + CATEGORY_FEATURE_NAMES
    numeric = np.empty((len(rows), len(numeric_names)), dtype=np.float32)
    cat_values = {name: np.empty(len(rows), dtype=object) for name in ITEM_CAT_FEATURES}

    iterator = tqdm(rows.itertuples(index=False), total=len(rows), desc=desc)
    for i, row in enumerate(iterator):
        features: dict[str, float | str] = pair_features(terms[row.term_id], items[row.item_id])
        features.update(category_pair_features(row.term_id, row.item_id, term_category_lookup, item_meta_lookup))
        numeric[i] = [float(features.get(name, 0.0)) for name in numeric_names]
        for name in ITEM_CAT_FEATURES:
            cat_values[name][i] = features.get(name, "unknown") or "unknown"

    frame = pd.DataFrame(numeric, columns=numeric_names)
    for name in ITEM_CAT_FEATURES:
        frame[name] = cat_values[name]
    return frame[FEATURE_NAMES]


def sample_frame(frame: pd.DataFrame, mask: pd.Series, n: int, seed: int) -> pd.DataFrame:
    subset = frame.loc[mask]
    if len(subset) <= n:
        return subset
    return subset.sample(n=n, random_state=seed)


def pair_key(frame: pd.DataFrame) -> pd.Series:
    return frame["term_id"].astype(str) + "\t" + frame["item_id"].astype(str)


def load_train_term_negatives(path: Path, positives: pd.DataFrame) -> pd.DataFrame:
    negatives = pd.read_csv(path, usecols=["term_id", "item_id"], dtype=str, keep_default_na=False)
    negatives = negatives.drop_duplicates(["term_id", "item_id"]).reset_index(drop=True)
    positive_keys = set(pair_key(positives))
    keep = ~pair_key(negatives).isin(positive_keys)
    dropped = int((~keep).sum())
    negatives = negatives.loc[keep].reset_index(drop=True)
    if negatives.empty:
        raise ValueError(f"No usable negatives found in {path}")
    if dropped:
        print(f"dropped {dropped:,} negatives that overlap known positives")
    print(f"using train-term negatives from {path} rows={len(negatives):,}")
    return negatives


def build_unlabeled_negatives(args: argparse.Namespace) -> pd.DataFrame:
    negative_path = Path(args.negatives) if args.negatives else None
    if negative_path and negative_path.exists():
        positives = pd.read_csv(Path(args.data_dir) / "training_pairs.csv", usecols=["term_id", "item_id"], dtype=str)
        return load_train_term_negatives(negative_path, positives)

    if not args.allow_submission_negatives:
        raise FileNotFoundError(
            f"Train-term negatives were not found at {negative_path}. "
            "Create train-term negatives first, or pass "
            "`--allow-submission-negatives` to use the older submission-pair fallback."
        )

    data_dir = Path(args.data_dir)
    pairs = pd.read_csv(data_dir / "submission_pairs.csv", usecols=["id", "term_id", "item_id"])
    scores = pd.read_csv(args.lexical_scores, usecols=["score"])
    if len(pairs) != len(scores):
        raise ValueError("submission_pairs and lexical_scores row counts differ")

    pairs["score"] = scores["score"].astype("float32").to_numpy()
    parts = [
        sample_frame(pairs, pairs["score"] < args.low_max, args.n_low, args.seed),
        sample_frame(
            pairs,
            (pairs["score"] >= args.low_max) & (pairs["score"] < args.mid_max),
            args.n_mid,
            args.seed + 1,
        ),
        sample_frame(
            pairs,
            (pairs["score"] >= args.mid_max) & (pairs["score"] < args.hard_max),
            args.n_hard,
            args.seed + 2,
        ),
    ]
    negatives = pd.concat(parts, ignore_index=True).drop_duplicates("id")
    print("negative score distribution")
    print(negatives["score"].describe(percentiles=[0.1, 0.5, 0.9, 0.99]).to_string())
    return negatives[["term_id", "item_id"]]


def load_feature_context(args: argparse.Namespace):
    data_dir = Path(args.data_dir)
    terms = load_terms(data_dir / "terms.csv")
    items = load_items(data_dir / "items.csv")
    item_meta_lookup = build_item_meta_lookup(data_dir / "items.csv")

    topk_path = Path(args.term_category_topk)
    if not topk_path.exists():
        raise FileNotFoundError(
            f"term category top-k file not found: {topk_path}. "
            "Run the category-aware notebook first or pass --term-category-topk."
        )
    term_category_topk = pd.read_csv(topk_path)
    term_category_lookup = build_term_category_lookup(term_category_topk)
    return terms, items, term_category_lookup, item_meta_lookup


def train(args: argparse.Namespace) -> None:
    start_time = time.time()
    model_path = Path(args.model)
    model_path.parent.mkdir(parents=True, exist_ok=True)

    terms, items, term_category_lookup, item_meta_lookup = load_feature_context(args)
    positives = pd.read_csv(Path(args.data_dir) / "training_pairs.csv", usecols=["term_id", "item_id"])
    negatives = build_unlabeled_negatives(args)

    if args.n_pos and len(positives) > args.n_pos:
        positives = positives.sample(n=args.n_pos, random_state=args.seed)

    x_pos = rows_to_frame(positives, terms, items, term_category_lookup, item_meta_lookup, "positive features")
    x_neg = rows_to_frame(negatives, terms, items, term_category_lookup, item_meta_lookup, "negative features")
    x = pd.concat([x_pos, x_neg], ignore_index=True)
    y = np.concatenate([np.ones(len(x_pos), dtype=np.int8), np.zeros(len(x_neg), dtype=np.int8)])
    print(f"matrix={x.shape} positives={int(y.sum()):,} positive_rate={y.mean():.4f}")

    del x_pos, x_neg
    gc.collect()

    x_train, x_valid, y_train, y_valid = train_test_split(
        x,
        y,
        test_size=args.valid_size,
        random_state=args.seed,
        stratify=y,
    )
    del x
    gc.collect()

    model_params = {
        "loss_function": "Logloss",
        "eval_metric": "Logloss",
        "iterations": args.iterations,
        "learning_rate": args.learning_rate,
        "depth": args.depth,
        "l2_leaf_reg": args.l2_leaf_reg,
        "random_strength": args.random_strength,
        "bootstrap_type": "Bernoulli",
        "subsample": args.subsample,
        "auto_class_weights": "Balanced",
        "random_seed": args.seed,
        "thread_count": args.thread_count,
        "allow_writing_files": False,
        "verbose": args.verbose,
        "task_type": args.task_type,
        "devices": args.devices,
    }
    if args.task_type == "CPU":
        model_params["rsm"] = args.rsm
    elif args.rsm != 1.0:
        print("CatBoost GPU ignores rsm for Logloss; training without rsm.")

    model = CatBoostClassifier(**model_params)

    train_pool = Pool(x_train, y_train, cat_features=CAT_FEATURES)
    valid_pool = Pool(x_valid, y_valid, cat_features=CAT_FEATURES)
    model.fit(train_pool, eval_set=valid_pool, use_best_model=False)

    valid_prob = model.predict_proba(valid_pool)[:, 1]
    best = (0.0, 0.5)
    for threshold in np.linspace(0.05, 0.95, 91):
        pred = (valid_prob >= threshold).astype(np.int8)
        score = f1_score(y_valid, pred, average="macro")
        if score > best[0]:
            best = (float(score), float(threshold))
    print(f"synthetic valid macro_f1={best[0]:.5f} threshold={best[1]:.3f}")

    importances = pd.Series(model.get_feature_importance(valid_pool), index=FEATURE_NAMES).sort_values(ascending=False)
    print(importances.head(35).to_string())

    model.save_model(str(model_path))
    joblib.dump(
        {
            "features": FEATURE_NAMES,
            "cat_features": CAT_FEATURES,
            "threshold": best[1],
            "iterations": args.iterations,
            "term_category_topk": str(args.term_category_topk),
        },
        model_path.with_suffix(".meta.joblib"),
    )
    elapsed = time.time() - start_time
    print(f"wrote {model_path}")
    print(f"elapsed_minutes={elapsed / 60:.1f}")


def predict(args: argparse.Namespace) -> None:
    data_dir = Path(args.data_dir)
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    model = CatBoostClassifier()
    model.load_model(str(args.model))
    terms, items, term_category_lookup, item_meta_lookup = load_feature_context(args)

    written = 0
    with output_path.open("w", newline="", encoding="utf-8") as out_handle:
        writer = csv.writer(out_handle)
        writer.writerow(["id", "term_id", "prob"])

        reader = pd.read_csv(data_dir / "submission_pairs.csv", chunksize=args.chunk_size)
        for chunk in tqdm(reader, desc="predict chunks"):
            matrix = rows_to_frame(chunk, terms, items, term_category_lookup, item_meta_lookup, "chunk features")
            pool = Pool(matrix, cat_features=CAT_FEATURES)
            prob = model.predict_proba(pool)[:, 1]
            writer.writerows(
                (row_id, term_id, f"{score:.8f}")
                for row_id, term_id, score in zip(chunk["id"], chunk["term_id"], prob)
            )
            written += len(chunk)
            del matrix, pool, prob, chunk
            gc.collect()

    print(f"wrote {output_path} rows={written:,}")


def make_submission(args: argparse.Namespace) -> None:
    scores = pd.read_csv(args.scores)
    if args.rate is not None:
        threshold = scores["prob"].quantile(1.0 - args.rate)
    else:
        threshold = args.threshold
    pred = (scores["prob"] >= threshold).astype("int8")
    output = pd.DataFrame({"id": scores["id"], "prediction": pred})
    out_path = Path(args.output)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    output.to_csv(out_path, index=False)
    print(f"threshold={threshold:.8f}")
    print(f"wrote {out_path}")
    print(f"rows={len(output):,} positives={int(pred.sum()):,} rate={pred.mean():.5f}")


def main() -> None:
    parser = argparse.ArgumentParser()
    sub = parser.add_subparsers(dest="cmd", required=True)

    common = argparse.ArgumentParser(add_help=False)
    common.add_argument("--data-dir", default="data")
    common.add_argument("--term-category-topk", default="outputs/category_aware_notebook/term_category_topk.csv")

    train_cmd = sub.add_parser("train", parents=[common])
    train_cmd.add_argument("--lexical-scores", default="outputs/lexical_scores.csv")
    train_cmd.add_argument("--negatives", default="outputs/train_term_negatives.csv")
    train_cmd.add_argument("--allow-submission-negatives", action="store_true")
    train_cmd.add_argument("--model", default="outputs/catboost_category_aware.cbm")
    train_cmd.add_argument("--n-pos", type=int, default=0)
    train_cmd.add_argument("--n-low", type=int, default=350_000)
    train_cmd.add_argument("--n-mid", type=int, default=350_000)
    train_cmd.add_argument("--n-hard", type=int, default=150_000)
    train_cmd.add_argument("--low-max", type=float, default=2.0)
    train_cmd.add_argument("--mid-max", type=float, default=4.5)
    train_cmd.add_argument("--hard-max", type=float, default=6.5)
    train_cmd.add_argument("--valid-size", type=float, default=0.15)
    train_cmd.add_argument("--iterations", type=int, default=5_000)
    train_cmd.add_argument("--learning-rate", type=float, default=0.025)
    train_cmd.add_argument("--depth", type=int, default=6)
    train_cmd.add_argument("--l2-leaf-reg", type=float, default=6.0)
    train_cmd.add_argument("--random-strength", type=float, default=0.5)
    train_cmd.add_argument("--subsample", type=float, default=0.85)
    train_cmd.add_argument("--rsm", type=float, default=0.95)
    train_cmd.add_argument("--thread-count", type=int, default=-1)
    train_cmd.add_argument("--verbose", type=int, default=100)
    train_cmd.add_argument("--task-type", choices=["CPU", "GPU"], default="CPU")
    train_cmd.add_argument("--devices", default="0")
    train_cmd.add_argument("--seed", type=int, default=42)
    train_cmd.set_defaults(func=train)

    pred_cmd = sub.add_parser("predict", parents=[common])
    pred_cmd.add_argument("--model", default="outputs/catboost_category_aware.cbm")
    pred_cmd.add_argument("--output", default="outputs/catboost_category_aware_scores.csv")
    pred_cmd.add_argument("--chunk-size", type=int, default=150_000)
    pred_cmd.set_defaults(func=predict)

    submit_cmd = sub.add_parser("submit")
    submit_cmd.add_argument("--scores", default="outputs/catboost_category_aware_scores.csv")
    submit_cmd.add_argument("--output", default="outputs/catboost_category_aware.csv")
    submit_cmd.add_argument("--threshold", type=float, default=0.5)
    submit_cmd.add_argument("--rate", type=float)
    submit_cmd.set_defaults(func=make_submission)

    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
import lexical_baseline
import train_term_negatives
import pu_catboost

importlib.reload(lexical_baseline)
importlib.reload(train_term_negatives)
importlib.reload(pu_catboost)

print("source files ready:", sorted(p.name for p in SRC_DIR.glob("*.py")))

## Build Query-Category Priors

`pu_catboost.py` uses a `term_category_topk.csv` file. This cell trains lightweight character TF-IDF + SGD category classifiers from positive training pairs and predicts top categories for every term.


In [ ]:
import math
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from tqdm.auto import tqdm

from lexical_baseline import normalize_text
from pu_catboost import add_category_levels

SEED = 42
CATEGORY_LEVELS = ("cat_l1", "cat_l2")
CATEGORY_TOPK = 5
CATEGORY_MAX_FEATURES = 220_000
CATEGORY_MAX_ITER = 35
TERM_CATEGORY_TOPK_PATH = CAT_OUT_DIR / "term_category_topk.csv"

terms_df = pd.read_csv(DATA_DIR / "terms.csv", dtype=str, keep_default_na=False)
items_meta = pd.read_csv(
    DATA_DIR / "items.csv",
    usecols=["item_id", "category"],
    dtype=str,
    keep_default_na=False,
)
training_pairs = pd.read_csv(
    DATA_DIR / "training_pairs.csv", dtype=str, keep_default_na=False
)
if "label" in training_pairs.columns:
    training_pairs = training_pairs.loc[training_pairs["label"].astype(str).ne("0")]

terms_for_train = terms_df[["term_id", "query"]].copy()
terms_for_train["query_norm"] = terms_for_train["query"].map(normalize_text)
items_levels = add_category_levels(items_meta)
category_examples = (
    training_pairs[["term_id", "item_id"]]
    .merge(terms_for_train[["term_id", "query_norm"]], on="term_id", how="left")
    .merge(items_levels[["item_id", "cat_l1", "cat_l2"]], on="item_id", how="left")
    .dropna(subset=["query_norm", "cat_l1", "cat_l2"])
    .drop_duplicates(["query_norm", "cat_l1", "cat_l2"])
    .reset_index(drop=True)
)
print("category examples =", len(category_examples))

vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=2,
    max_features=CATEGORY_MAX_FEATURES,
    dtype=np.float32,
    sublinear_tf=True,
)
x_all = vectorizer.fit_transform(category_examples["query_norm"])
print("category tfidf shape =", x_all.shape)

models = {}
for level in CATEGORY_LEVELS:
    counts = category_examples[level].value_counts()
    keep_labels = counts.index[counts >= 2]
    keep = category_examples[level].isin(keep_labels).to_numpy()
    x = x_all[keep]
    y = category_examples.loc[keep, level].to_numpy()
    clf = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=2e-5,
        max_iter=CATEGORY_MAX_ITER,
        tol=1e-3,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    )
    start = time.time()
    clf.fit(x, y)
    models[level] = clf
    print(
        f"{level}: classes={len(clf.classes_):,} examples={len(y):,} seconds={time.time() - start:.1f}"
    )

term_ids = terms_df["term_id"].astype(str).to_numpy()
queries = terms_df["query"].map(normalize_text).to_numpy()
chunks = []
for start in tqdm(range(0, len(terms_df), 25_000), desc="category topk"):
    end = min(start + 25_000, len(terms_df))
    x = vectorizer.transform(queries[start:end])
    out = {"term_id": term_ids[start:end]}
    for level, clf in models.items():
        proba = clf.predict_proba(x)
        k = min(CATEGORY_TOPK, proba.shape[1])
        idx = np.argpartition(proba, -k, axis=1)[:, -k:]
        order = np.argsort(-np.take_along_axis(proba, idx, axis=1), axis=1)
        idx = np.take_along_axis(idx, order, axis=1)
        scores = np.take_along_axis(proba, idx, axis=1).astype(np.float32)
        labels = clf.classes_[idx]
        for rank in range(k):
            out[f"{level}_top{rank + 1}"] = labels[:, rank]
            out[f"{level}_prob{rank + 1}"] = scores[:, rank]
    chunks.append(pd.DataFrame(out))

term_category_topk = pd.concat(chunks, ignore_index=True)
term_category_topk.to_csv(TERM_CATEGORY_TOPK_PATH, index=False)
print("wrote", TERM_CATEGORY_TOPK_PATH, "rows=", len(term_category_topk))
term_category_topk.head()

## Mine Negatives From Training Terms

This creates `/kaggle/working/outputs/train_term_negatives.csv`. Defaults produce about 870k negatives on the provided data.


In [ ]:
from train_term_negatives import mine_negatives

NEGATIVES_PATH = OUT_DIR / "train_term_negatives.csv"
negative_args = argparse.Namespace(
    data_dir=str(DATA_DIR),
    output=str(NEGATIVES_PATH),
    hard_per_term=25,
    mid_per_term=15,
    easy_per_term=10,
    retrieve_topk=300,
    chunk_size=128,
    max_features=250_000,
    min_df=2,
    word_ngram_max=2,
    seed=42,
    limit_terms=None,
    limit_items=None,
)
mine_negatives(negative_args)

neg = pd.read_csv(NEGATIVES_PATH)
print("negative rows =", len(neg))
print(neg["negative_band"].value_counts().to_string())

## Train CatBoost On GPU

Start with 3000 iterations. The previous 10k+ iteration run was likely overfitting the synthetic task; fix the negative source first, then tune capacity.


In [ ]:
import importlib
import pu_catboost

pu_catboost = importlib.reload(pu_catboost)

MODEL_PATH = OUT_DIR / "catboost_train_term_negatives.cbm"
train_args = argparse.Namespace(
    data_dir=str(DATA_DIR),
    term_category_topk=str(TERM_CATEGORY_TOPK_PATH),
    lexical_scores=str(OUT_DIR / "unused_lexical_scores.csv"),
    negatives=str(NEGATIVES_PATH),
    allow_submission_negatives=False,
    model=str(MODEL_PATH),
    n_pos=0,
    n_low=350_000,
    n_mid=350_000,
    n_hard=150_000,
    low_max=2.0,
    mid_max=4.5,
    hard_max=6.5,
    valid_size=0.15,
    iterations=3000,
    learning_rate=0.025,
    depth=6,
    l2_leaf_reg=8.0,
    random_strength=0.8,
    subsample=0.85,
    rsm=1.0,
    thread_count=-1,
    verbose=100,
    task_type="GPU",
    devices="0",
    seed=42,
)
pu_catboost.train(train_args)

## Predict Submission Pairs


In [ ]:
import importlib
import pu_catboost

pu_catboost = importlib.reload(pu_catboost)

SCORES_PATH = OUT_DIR / "catboost_train_term_negatives_scores.csv"
predict_args = argparse.Namespace(
    data_dir=str(DATA_DIR),
    term_category_topk=str(TERM_CATEGORY_TOPK_PATH),
    model=str(MODEL_PATH),
    output=str(SCORES_PATH),
    chunk_size=150_000,
)
pu_catboost.predict(predict_args)

scores_preview = pd.read_csv(SCORES_PATH, nrows=5)
scores_preview

## Write Rate-Based Submissions

Submit one or two files first. Historically the useful zone was around 18-22 percent positives, so start with `r20` or `r22`.


In [ ]:
import importlib
import pu_catboost

pu_catboost = importlib.reload(pu_catboost)

SUBMISSION_PATHS = []
for rate in [0.18, 0.20, 0.22, 0.24]:
    out_path = WORK_DIR / f"submission_catboost_trainneg_r{int(rate * 100):02d}.csv"
    submit_args = argparse.Namespace(
        scores=str(SCORES_PATH),
        output=str(out_path),
        threshold=0.5,
        rate=rate,
    )
    pu_catboost.make_submission(submit_args)
    SUBMISSION_PATHS.append(out_path)

print("submission files:")
for path in SUBMISSION_PATHS:
    print(path)

In [ ]:
# Final sanity check: each submission must have exactly id,prediction columns and match sample row count if sample_submission exists.
for path in SUBMISSION_PATHS:
    sub = pd.read_csv(path)
    print(
        path.name,
        sub.shape,
        sub["prediction"].sum(),
        sub["prediction"].mean(),
        list(sub.columns),
    )
    assert list(sub.columns) == ["id", "prediction"]
    assert set(sub["prediction"].unique()).issubset({0, 1})

sample_path = DATA_DIR / "sample_submission.csv"
if sample_path.exists():
    sample = pd.read_csv(sample_path, usecols=["id"])
    for path in SUBMISSION_PATHS:
        sub = pd.read_csv(path, usecols=["id"])
        assert len(sub) == len(sample), f"row count mismatch for {path}"
        assert (
            sub["id"].astype(str).equals(sample["id"].astype(str))
        ), f"id order mismatch for {path}"
print("sanity checks passed")